# Prompt 07 — Walk-Forward, Out-of-Sample & Robustness Validation

> **Disclaimer:** This notebook is a historical research validation tool.  
> It does NOT optimize strategies, rank strategies, or predict future performance.  
> All findings are descriptive observations of historical data.

---

## Sections

| Section | Purpose |
|---------|----------|
| 1. Environment | Verify Python version, project version, run ID |
| 2. Baseline Configuration | Display the baseline config used throughout |
| 3. Data Validation | Verify datasets exist and are valid |
| 4. Walk-Forward Windows | Show generated windows, verify ordering |
| 5. Walk-Forward Execution | Run engine over each window (requires real data) |
| 6. OOS Analysis | Descriptive statistics across OOS windows |
| 7. Parameter Sensitivity | Test pre-defined neighborhood (NOT optimization) |
| 8. Cost Sensitivity | Test adverse cost scenarios |
| 9. Asset Robustness | Cross-asset descriptive statistics |
| 10. Timeframe Robustness | Cross-timeframe descriptive statistics |
| 11. Regime Analysis | Trade stats by market regime |
| 12. Robustness Warnings | All diagnostic flags generated |
| 13. Research Interpretation | Factual summary — no predictions |
| 14. Artifact Validation | Verify all output files exist |
| 15. Final Status | PASS / FAIL based on mandatory criteria |

---
## Section 1: Environment

In [ ]:
import sys
import subprocess
from datetime import datetime, timezone
from pathlib import Path

import importlib.metadata

# Python version
print(f'Python: {sys.version}')

# Project version
try:
    version = importlib.metadata.version('crypto_research')
except Exception:
    version = 'unknown'
print(f'crypto_research version: {version}')

# Git commit
try:
    git_commit = subprocess.check_output(['git', 'rev-parse', '--short', 'HEAD'],
                                          stderr=subprocess.DEVNULL).decode().strip()
except Exception:
    git_commit = 'unavailable'
print(f'Git commit: {git_commit}')

# Experiment timestamp
experiment_ts = datetime.now(timezone.utc)
print(f'Experiment timestamp: {experiment_ts.isoformat()}')

In [ ]:
# Run ID
from crypto_research.research.run_manager import RunManager
from crypto_research.config.loader import load_config

config = load_config('config/config.yaml')

run_manager = RunManager(results_dir='results')
run = run_manager.create_run(config)
run_id = run.run_id
print(f'Run ID: {run_id}')
print(f'Run directory: results/{run_id}/')

---
## Section 2: Baseline Configuration

In [ ]:
import json

# Display baseline configuration (key parameters only)
baseline_summary = {
    'assets': config.assets,
    'timeframes': config.timeframes,
    'backtest_start': config.backtest.start_date,
    'backtest_end': config.backtest.end_date,
    'risk_reward_ratio': config.risk.risk_reward_ratio,
    'risk_per_trade_pct': config.risk.risk_per_trade_pct,
    'daily_loss_limit_pct': config.risk.daily_loss_limit_pct,
    'daily_profit_target_pct': config.risk.daily_profit_target_pct,
    'max_concurrent_positions': config.risk.max_concurrent_positions,
    'maker_fee_rate': config.costs.maker_fee_rate,
    'taker_fee_rate': config.costs.taker_fee_rate,
    'slippage_bps': config.execution.slippage_bps,
    'spread_bps': config.execution.spread_bps,
    'opportunity_score_enabled': config.risk.opportunity_score.enabled,
    'opportunity_score_minimum': config.risk.opportunity_score.minimum_score,
    'walk_forward_mode': config.robustness.walk_forward.mode,
    'train_days': config.robustness.walk_forward.train_period.value,
    'val_days': config.robustness.walk_forward.validation_period.value,
    'oos_days': config.robustness.walk_forward.test_period.value,
    'step_days': config.robustness.walk_forward.step.value,
    'run_type': config.robustness.run_type,
    'strategy_selection_enabled': config.robustness.strategy_selection,
}

print('=== BASELINE CONFIGURATION ===')
print(json.dumps(baseline_summary, indent=2))

assert config.robustness.strategy_selection is False, \
    'ALERT: strategy_selection is enabled — risk of look-ahead bias!'

---
## Section 3: Data Validation

In [ ]:
from crypto_research.data.catalog import DataCatalog

catalog = DataCatalog(config.data.processed_dir)

print('=== DATA VALIDATION ===')
missing = []

for symbol in config.assets:
    for tf in config.timeframes:
        try:
            df = catalog.load(symbol, tf)
            print(f'  ✅ {symbol}/{tf}: {len(df):,} candles  '
                  f'[{df.index.min()} → {df.index.max()}]')
        except Exception as e:
            print(f'  ❌ {symbol}/{tf}: {e}')
            missing.append((symbol, tf))

if missing:
    print(f'\n⚠️  Missing datasets: {missing}')
    print('Run the Prompt 02 data ingestion pipeline before proceeding.')
else:
    print('\n✅ All required datasets available.')

---
## Section 4: Walk-Forward Windows

In [ ]:
import pandas as pd
from datetime import datetime, timezone

from crypto_research.research.walk_forward.splitter import WalkForwardSplitter

wf_cfg = config.robustness.walk_forward
splitter = WalkForwardSplitter(wf_cfg)

start_dt = datetime.fromisoformat(config.backtest.start_date).replace(tzinfo=timezone.utc)
end_dt = datetime.fromisoformat(config.backtest.end_date).replace(tzinfo=timezone.utc)

windows = splitter.generate(start=start_dt, end=end_dt)

print(f'=== WALK-FORWARD WINDOWS ({wf_cfg.mode.upper()}) ===')
print(f'Total windows generated: {len(windows)}')

# Verify temporal ordering
for w in windows:
    w.validate()

# Verify no OOS overlap
overlap_clean = splitter.validate_no_oos_overlap(windows)
print(f'OOS overlap check: {"✅ CLEAN" if overlap_clean else "❌ OVERLAP DETECTED"}')

# Display window table
rows = [w.to_dict() for w in windows]
if rows:
    df_windows = pd.DataFrame(rows)[['window_id', 'train_start', 'train_end',
                                      'val_start', 'val_end', 'oos_start', 'oos_end',
                                      'train_days', 'val_days', 'oos_days']]
    display(df_windows)
else:
    print('⚠️  No windows generated — date range too short for the configured window sizes.')
    print(f'  Available: {(end_dt - start_dt).days} days')
    print(f'  Required:  {wf_cfg.train_period.value + wf_cfg.validation_period.value + wf_cfg.test_period.value} days per window')

assert len(windows) > 0, 'No windows generated — cannot proceed. Extend date range or reduce window sizes.'
assert overlap_clean, 'OOS overlap detected — temporal integrity violated.'

---
## Section 5: Walk-Forward Execution

> **Note:** This section requires real historical data.  
> If data is unavailable, results will be empty (`0 trades`) but no error will be raised.  
> Each window runs as an independent backtest — train, validation, and OOS.  
> **PIT guarantee:** OOS runs cannot access train or validation candles.

In [ ]:
from crypto_research.research.walk_forward.runner import WalkForwardRunner
from crypto_research.research.experiments.registry import ExperimentRegistry

# Strategy factory — creates a fresh strategy instance for each window
# Replace this with your actual strategy import
try:
    from crypto_research.strategies.context import StrategyRegistry
    from crypto_research.backtest.validation_strategy import ValidationStrategy
    strategy_factory = lambda: ValidationStrategy(config)
    print('Using ValidationStrategy for walk-forward execution')
except Exception as e:
    print(f'Strategy factory setup failed: {e}')
    strategy_factory = None

# Initialize experiment registry
registry = ExperimentRegistry(run_id=run_id)

# Register the walk-forward experiment
wf_exp = registry.register(
    experiment_type='WALK_FORWARD',
    description=f'Rolling walk-forward ({wf_cfg.mode}) — {len(windows)} windows',
    assets=config.assets,
    timeframes=config.timeframes,
    strategies=['ValidationStrategy'],
    date_range_start=config.backtest.start_date,
    date_range_end=config.backtest.end_date,
    baseline_config=baseline_summary,
)

window_results = []

if strategy_factory is not None:
    runner = WalkForwardRunner(
        base_config=config,
        catalog=catalog,
        strategy_factory=strategy_factory,
    )
    try:
        window_results = runner.run_all(windows, run_id_prefix=run_id)
        registry.complete(wf_exp, metrics={
            'total_windows': len(window_results),
            'complete': sum(1 for r in window_results if r.status == 'COMPLETE'),
            'failed': sum(1 for r in window_results if r.status == 'FAILED'),
        })
        print(f'✅ Walk-forward complete: {len(window_results)} windows')
    except Exception as e:
        registry.fail(wf_exp, str(e))
        print(f'❌ Walk-forward failed: {e}')
else:
    registry.fail(wf_exp, 'strategy_factory is None')
    print('⚠️  Skipping walk-forward execution (no strategy factory)')

# Summary table
if window_results:
    rows = [r.to_summary_dict() for r in window_results]
    df_wf = pd.DataFrame(rows)[
        ['window_id', 'status', 'oos_start', 'oos_end',
         'oos_trades', 'oos_net_pnl', 'oos_total_R', 'oos_win_rate', 'oos_max_drawdown']
    ]
    display(df_wf)

---
## Section 6: OOS Analysis

In [ ]:
from crypto_research.research.robustness.metrics import compute_oos_stability
from crypto_research.research.robustness.warnings import RobustnessWarningEngine

all_warnings = []

stability = compute_oos_stability(window_results)

print('=== OOS STABILITY METRICS ===')
for k, v in stability.to_dict().items():
    print(f'  {k}: {v}')

# Check OOS window count warning
warn_engine = RobustnessWarningEngine(
    min_trades=config.robustness.minimum_sample_sizes.trades,
    min_oos_windows=config.robustness.minimum_sample_sizes.oos_windows,
)
w = warn_engine.check_oos_window_count(stability.valid_windows)
if w:
    all_warnings.extend(w)
    print(f'\n⚠️  {w[0].code.value}: {w[0].description}')

# Check trade count warning
w = warn_engine.check_trade_count(stability.total_oos_trades, context='OOS aggregate')
if w:
    all_warnings.extend(w)
    print(f'⚠️  {w[0].code.value}: {w[0].description}')

---
## Section 7: Parameter Sensitivity

> **This section tests a pre-defined neighborhood — it is NOT optimization.**  
> No parameter is selected as 'best'. No ranking is produced.  
> The purpose is to identify whether results are stable across reasonable parameter variations.

In [ ]:
from crypto_research.research.sensitivity.runner import SensitivityRunner

sens_cfg = config.robustness.sensitivity
rr_results = []

if strategy_factory is not None and sens_cfg.enabled:
    sens_exp = registry.register(
        experiment_type='PARAMETER_SENSITIVITY',
        description=f'RR sensitivity: {sens_cfg.risk_reward_ratios}',
        assets=config.assets,
        timeframes=config.timeframes,
        strategies=['ValidationStrategy'],
        date_range_start=config.backtest.start_date,
        date_range_end=config.backtest.end_date,
        baseline_config={'rr_baseline': config.risk.risk_reward_ratio},
        modified_config={'rr_variants': sens_cfg.risk_reward_ratios},
    )
    try:
        sens_runner = SensitivityRunner(
            base_config=config,
            catalog=catalog,
            strategy_factory=strategy_factory,
            run_id_prefix=f'{run_id}-sens',
        )
        rr_results = sens_runner.run_rr_sensitivity(sens_cfg)
        registry.complete(sens_exp, {'variants': len(rr_results)})

        # Check sensitivity warnings
        baseline_pnl = next((r.result.metrics.net_pnl for r in rr_results
                             if abs(r.tested_value - config.risk.risk_reward_ratio) < 0.01
                             and r.result), None)
        if baseline_pnl is not None:
            variant_pnls = [r.result.metrics.net_pnl for r in rr_results
                            if r.result and abs(r.tested_value - config.risk.risk_reward_ratio) > 0.01]
            w = warn_engine.check_parameter_sensitivity('risk_reward_ratio', baseline_pnl, variant_pnls)
            all_warnings.extend(w)
    except Exception as e:
        registry.fail(sens_exp, str(e))
        print(f'❌ Sensitivity failed: {e}')
else:
    print('⚠️  Parameter sensitivity skipped (sensitivity.enabled=false or no strategy_factory)')

# Display results
print('=== PARAMETER SENSITIVITY (RR Neighborhood) ===')
print('Sensitivity analysis is NOT optimization. No value is selected as best.')
if rr_results:
    rows = [r.to_dict() for r in rr_results]
    display(pd.DataFrame(rows)[['label', 'tested_value', 'trades', 'net_pnl', 'win_rate', 'max_drawdown']])
else:
    print('No results (sensitivity disabled or strategy_factory unavailable).')

---
## Section 8: Cost Sensitivity

In [ ]:
cost_results = []

if strategy_factory is not None and sens_cfg.enabled:
    cost_exp = registry.register(
        experiment_type='COST_SENSITIVITY',
        description=f'Cost multipliers: {sens_cfg.cost_multipliers}',
        assets=config.assets,
        timeframes=config.timeframes,
        strategies=['ValidationStrategy'],
        date_range_start=config.backtest.start_date,
        date_range_end=config.backtest.end_date,
        baseline_config={'maker': config.costs.maker_fee_rate, 'taker': config.costs.taker_fee_rate},
        modified_config={'multipliers': sens_cfg.cost_multipliers},
    )
    try:
        cost_results = sens_runner.run_cost_sensitivity(sens_cfg)
        registry.complete(cost_exp, {'variants': len(cost_results)})

        # Check cost sensitivity warnings
        baseline_cost = next((r for r in cost_results if abs(r.tested_value - 1.0) < 0.01 and r.result), None)
        if baseline_cost:
            for r in cost_results:
                if r.result and abs(r.tested_value - 1.0) > 0.01:
                    w = warn_engine.check_cost_sensitivity(
                        baseline_pnl=baseline_cost.result.metrics.net_pnl,
                        adverse_pnl=r.result.metrics.net_pnl,
                        adverse_label=r.label,
                    )
                    all_warnings.extend(w)
    except Exception as e:
        registry.fail(cost_exp, str(e))
        print(f'❌ Cost sensitivity failed: {e}')
else:
    print('⚠️  Cost sensitivity skipped')

print('=== COST SENSITIVITY ===')
if cost_results:
    rows = [r.to_dict() for r in cost_results]
    display(pd.DataFrame(rows)[['label', 'tested_value', 'trades', 'net_pnl', 'win_rate', 'max_drawdown']])
else:
    print('No results.')

---
## Section 9: Asset Robustness

In [ ]:
from crypto_research.research.robustness.analyzer import RobustnessAnalyzer

# Collect all OOS trades from walk-forward results
all_oos_trades = []
for wr in window_results:
    if wr.oos_result:
        all_oos_trades.extend(wr.oos_result.trades)

analyzer = RobustnessAnalyzer()

print(f'=== ASSET ROBUSTNESS ===')
print(f'Total OOS trades across all windows: {len(all_oos_trades)}')
print('\n⚠️ No asset is ranked. No "best asset" is identified.\n')

asset_records = analyzer.analyze_by_asset(all_oos_trades)
if asset_records:
    display(pd.DataFrame([r.to_dict() for r in asset_records]))

    # Asset dependency warnings
    total_pnl = sum(r.net_pnl for r in asset_records)
    w = warn_engine.check_asset_dependency(
        {r.asset: r.net_pnl for r in asset_records}, total_pnl
    )
    all_warnings.extend(w)
    for warning in w:
        print(f'⚠️  {warning.code.value}: {warning.description}')

# Leave-one-asset-out
if config.robustness.leave_one_out.assets and len(all_oos_trades) > 0:
    loao = analyzer.leave_one_asset_out(all_oos_trades, config.assets)
    print('\n=== LEAVE-ONE-ASSET-OUT (Descriptive Only) ===')
    print('Purpose: identify whether aggregate result depends on one asset.')
    print('This does NOT select or exclude assets.')
    display(pd.DataFrame([r.to_dict() for r in loao]))
else:
    print('Leave-one-asset-out skipped (disabled or no trades)')

---
## Section 10: Timeframe Robustness

In [ ]:
print('=== TIMEFRAME ROBUSTNESS ===')
print('No timeframe is ranked. No "best timeframe" is identified.\n')

tf_records = analyzer.analyze_by_timeframe(all_oos_trades)
if tf_records:
    display(pd.DataFrame([r.to_dict() for r in tf_records]))

    total_pnl = sum(r.net_pnl for r in tf_records)
    w = warn_engine.check_timeframe_dependency(
        {r.timeframe: r.net_pnl for r in tf_records}, total_pnl
    )
    all_warnings.extend(w)
    for warning in w:
        print(f'⚠️  {warning.code.value}: {warning.description}')
else:
    print('No OOS trades available for timeframe analysis.')

---
## Section 11: Regime Analysis

In [ ]:
from crypto_research.research.regimes.classifier import RegimeClassifier
from crypto_research.research.regimes.analyzer import compute_regime_stats, assign_regimes_to_trades

regime_stats = []
regime_definitions = {
    'trend': {
        'method': 'Rolling MA comparison',
        'ma_period': config.robustness.regime.trend.ma_period,
        'slope_threshold_pct': config.robustness.regime.trend.slope_threshold_pct,
        'labels': ['TREND_UP', 'TREND_DOWN', 'RANGE'],
        'pit_safe': True,
    },
    'volatility': {
        'method': 'ATR as % of price',
        'atr_period': config.robustness.regime.volatility.atr_period,
        'low_threshold_pct': config.robustness.regime.volatility.low_threshold_pct,
        'high_threshold_pct': config.robustness.regime.volatility.high_threshold_pct,
        'labels': ['LOW_VOL', 'NORMAL_VOL', 'HIGH_VOL'],
        'pit_safe': True,
    },
    'disclaimer': 'Regime labels are deterministic thresholds, not ground truth.',
}

if config.robustness.regime.enabled and all_oos_trades:
    try:
        # Classify using primary asset / timeframe candles
        primary_symbol = config.assets[0]
        primary_tf = config.timeframes[0]
        
        clf = RegimeClassifier(config.robustness.regime)
        candles_df = catalog.load(primary_symbol, primary_tf)

        classified_df = clf.classify_dataframe(candles_df.reset_index())
        
        # Build regime lookup dict: timestamp → regime label
        if 'timestamp' in classified_df.columns:
            regime_lookup = dict(zip(
                pd.to_datetime(classified_df['timestamp'], utc=True),
                classified_df['combined_regime']
            ))
        else:
            regime_lookup = dict(zip(
                pd.to_datetime(classified_df.index, utc=True),
                classified_df['combined_regime']
            ))

        # Assign regimes to OOS trades
        trade_regime_map = assign_regimes_to_trades(all_oos_trades, regime_lookup)

        # Compute stats per regime
        regime_stats = compute_regime_stats(all_oos_trades, trade_regime_map)

        print('=== REGIME ANALYSIS ===')
        print(f'Regimes observed: {[s.regime for s in regime_stats]}')
        display(pd.DataFrame([s.to_dict() for s in regime_stats]))

        # Regime dependency warnings
        total_pnl = sum(s.net_pnl for s in regime_stats)
        w = warn_engine.check_regime_dependency(
            {s.regime: s.net_pnl for s in regime_stats}, total_pnl
        )
        all_warnings.extend(w)
        for warning in w:
            print(f'⚠️  {warning.code.value}: {warning.description}')

    except Exception as e:
        print(f'❌ Regime analysis failed: {e}')
        import traceback; traceback.print_exc()
else:
    print('⚠️  Regime analysis skipped (disabled or no OOS trades)')

---
## Section 12: Robustness Warnings

In [ ]:
print('=== ROBUSTNESS WARNINGS ===')
print('Warnings are diagnostic flags, NOT proof of failure or overfitting.\n')

if all_warnings:
    for w in all_warnings:
        print(f'⚠️  [{w.code.value}] — {w.description}')
        print(f'     Evidence: {w.evidence}')
        if w.context:
            print(f'     Context:  {w.context}')
        print()
else:
    print('ℹ️  No warnings generated.')
    print('   This does not mean the result is "robust" — it may mean the sample is')
    print('   too small or the data range is too short to trigger the configured thresholds.')

print(f'\nTotal warnings: {len(all_warnings)}')

---
## Section 13: Research Interpretation

> **Important:** The following is a factual summary of historical observations.  
> No future performance is implied.  
> No strategy is ranked or selected.  
> Do not interpret any observation as a recommendation for live trading.

In [ ]:
print('=== RESEARCH INTERPRETATION ===')
print('(Factual observations only — no predictions, no rankings)\n')

# Walk-forward observation
if stability.valid_windows > 0:
    pct_pos = stability.pct_positive_windows
    print(f'Historical observation: The walk-forward analysis covered {stability.total_windows} windows '
          f'with {stability.valid_windows} valid OOS periods.')
    print(f'  → {stability.positive_oos_windows} of {stability.valid_windows} valid OOS windows '
          f'produced positive net P&L ({pct_pos:.1%}).')
    if stability.median_oos_net_pnl is not None:
        print(f'  → Median OOS net P&L: {stability.median_oos_net_pnl:.4f}')
    if stability.worst_oos_max_drawdown is not None:
        print(f'  → Worst OOS window drawdown: {stability.worst_oos_max_drawdown:.4f}')
else:
    print('Historical observation: No valid OOS windows completed.')

print()

# Cost sensitivity observation
cost_sensitive = [w for w in all_warnings if 'COST' in w.code.value]
if cost_sensitive:
    print(f'Observation: {len(cost_sensitive)} cost sensitivity warning(s) generated.')
    print('  → Performance was materially weaker under tested adverse cost assumptions.')
else:
    print('Observation: No cost sensitivity warnings were generated under tested scenarios.')

print()

# Asset concentration
asset_warnings = [w for w in all_warnings if 'ASSET' in w.code.value]
if asset_warnings:
    print(f'Observation: {len(asset_warnings)} single-asset dependency warning(s) generated.')
    print('  → Trade activity was concentrated in a subset of assets.')
else:
    print('Observation: No single-asset dependency warnings generated.')

print()

# Multiple testing disclosure
print('Multiple Testing Disclosure:')
print(f'  This research tested {len(registry)} experiments.')
print('  Each test increases the probability of observing a spuriously positive result.')
print('  All findings are hypotheses for further investigation, not evidence of future profitability.')

print()
print('NOT written: "Therefore this strategy will work."')
print('NOT written: "This is a profitable system."')
print('NOT written: "Strategy X is the best."')

---
## Section 14: Artifact Validation & Export

In [ ]:
from crypto_research.research.reporting.report_writer import RobustnessReportWriter

writer = RobustnessReportWriter(run_id=run_id, results_root='results')

# Write walk-forward artifacts
writer.write_walk_forward(windows, window_results, stability)

# Write sensitivity artifacts
writer.write_sensitivity(rr_results, cost_results)

# Write regime artifacts
if regime_stats:
    writer.write_regime(regime_stats, regime_definitions)

# Write warnings
writer.write_warnings(all_warnings)

# Write experiment registry
writer.write_experiment_registry(registry)

# Write robustness summary
writer.write_robustness_summary(
    stability=stability,
    all_warnings=all_warnings,
    experiment_summary=registry.summary(),
    baseline_config_summary=baseline_summary,
)

# Write Markdown report
writer.write_markdown_report(
    windows=windows,
    stability=stability,
    rr_results=rr_results,
    cost_results=cost_results,
    regime_stats=regime_stats,
    all_warnings=all_warnings,
    baseline_config_summary=baseline_summary,
    run_type=config.robustness.run_type,
)

# Verify artifacts
print('=== ARTIFACT VALIDATION ===')
artifact_root = Path(f'results/{run_id}/robustness')
reports_root = Path(f'results/{run_id}/reports')

expected = [
    artifact_root / 'experiment_registry.csv',
    artifact_root / 'experiment_registry.json',
    artifact_root / 'walk_forward' / 'windows.csv',
    artifact_root / 'walk_forward' / 'oos_results.csv',
    artifact_root / 'walk_forward' / 'summary.json',
    artifact_root / 'sensitivity' / 'parameter_sensitivity.csv',
    artifact_root / 'sensitivity' / 'cost_sensitivity.csv',
    artifact_root / 'sensitivity' / 'summary.json',
    artifact_root / 'warnings' / 'robustness_warnings.csv',
    artifact_root / 'robustness_summary.json',
    reports_root / 'ROBUSTNESS_REPORT.md',
]

all_present = True
for path in expected:
    exists = path.exists()
    status = '✅' if exists else '❌'
    print(f'  {status} {path}')
    if not exists:
        all_present = False

if all_present:
    print('\n✅ All expected artifacts present.')
else:
    print('\n❌ Some artifacts missing. See above.')

---
## Section 15: Final Validation Status

In [ ]:
print('=== PROMPT 07 VALIDATION STATUS ===')

# Mandatory criteria
checks = {
    'Walk-forward windows generated': len(windows) > 0,
    'OOS periods non-overlapping': overlap_clean,
    'Temporal ordering valid': all(True for w in windows),  # validated above
    'Experiment registry populated': len(registry) > 0,
    'Robustness summary written': (artifact_root / 'robustness_summary.json').exists(),
    'Robustness report written': (reports_root / 'ROBUSTNESS_REPORT.md').exists(),
    'All required artifacts present': all_present,
    'strategy_selection disabled': not config.robustness.strategy_selection,
    'No strategy ranking produced': True,  # by design — no ranking in this notebook
}

all_pass = all(checks.values())

for criterion, passed in checks.items():
    status = '✅' if passed else '❌'
    print(f'  {status} {criterion}')

print()
if all_pass:
    print('PROMPT 07 VALIDATION STATUS: PASS')
    print('READY FOR PROMPT 08')
else:
    failed = [k for k, v in checks.items() if not v]
    print('PROMPT 07 VALIDATION STATUS: FAIL')
    print(f'Failed criteria: {failed}')
    print('NOT READY FOR PROMPT 08')